In [39]:
import networkx as nx
import numpy as np
from tqdm import trange 

def create_toy_graph():
    G = nx.DiGraph()
    G.add_nodes_from([i for i in range(4)])
    G.add_edges_from([(0, 1), (0,2), (1,3), (2, 3)])
    for e in G.edges():
        G.edges()[e]['w'] = 1
        G.edges()[e]['class'] = e
    return G

def run_sim(G, N=2000):

    seq = np.random.SeedSequence(123)
    mat = np.zeros((N, 2))
    paths = list(nx.all_simple_edge_paths(G, 0, 3))

    
    for i in trange(N):
        rng = np.random.default_rng(seq.spawn(1)[0])
        for _,_, d in G.edges(data=True):
            d['w_real'] = rng.exponential(d['w'])
        length_paths = [[G.edges()[p]['w_real'] for p in path] for path in paths]
        mat[i, :] = [sum(p) for p in length_paths]
        shortest_time = list(np.min(mat, axis=1))
    return shortest_time

G = create_toy_graph()
list(set(d['class'] for _,_,d in G.edges(data=True)))

[(0, 1), (0, 2), (2, 3), (1, 3)]

## Original Design:

In [ ]:
def run_sim(G, s, t, N=20_000, step_size=1e-6):
    seq = np.random.SeedSequence(123)
    num_edges = len(G.edges())
    paths = list(nx.all_simple_edge_paths(G, s, t))
    mat = np.zeros((N, len(paths), num_edges+1))

    for i in trange(N):
        rng = np.random.default_rng(seq.spawn(1)[0])
        nx.set_edge_attributes(G, {(u,v): rng.exponential(d['w']) for u,v,d in G.edges(data=True)}, name='w_realized')
        length_paths = [[G.edges()[p]['w_realized'] for p in path] for path in paths]
        mat[i, :, 0] = [sum(p) for p in length_paths]
        for j, params in enumerate(G.edges(data=True)):
            u,v,d = params
            d['w_realized'] += step_size
            length_paths = [[G.edges()[p]['w_realized'] for p in path] for path in paths]
            mat[i, :, j+1] = [sum(p) for p in length_paths]
            d['w_realized'] -= step_size
    shortest_time = np.min(mat, axis=1)
    dell_mat = (shortest_time[:, 1:] - shortest_time[:, [0]])/step_size
    return dell_mat, shortest_time[:, [0]]

run_sim(G, 0, 3)


100%|██████████| 20000/20000 [00:02<00:00, 9686.02it/s] 


(array([[1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [1., 0., 1., 0.],
        ...,
        [1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [0., 1., 0., 1.]]),
 array([[0.85381172],
        [1.47819433],
        [1.65662586],
        ...,
        [4.57410533],
        [2.28127969],
        [1.13166428]]))

## Proposed Design:

In [ ]:
from collections import defaultdict

def new(G, s=0, t=3, attribute_name='class', N=20_000, step_size=1e-6):
    seq = np.random.SeedSequence(123)
    class_edges = defaultdict(list)
    for u,v,d in G.edges(data=True):
        class_edges[d[attribute_name]].append((u,v))
    mat = np.zeros((N, len(class_edges)+1))
    for i in trange(N):
        rng = np.random.default_rng(seq.spawn(1)[0])
        nx.set_edge_attributes(G, {(u,v): rng.exponential(d['w']) for u,v,d in G.edges(data=True)}, name='w_realized')
        mat[i, 0] = nx.shortest_path_length(G, s, t, weight='w_realized')
        for j, (clss, edges) in enumerate(class_edges.items()):
            for u, v in edges:
                G[u][v]['w_realized'] += step_size
            mat[i, j+1] = nx.shortest_path_length(G, s, t, weight='w_realized')
            for u, v in edges:
                G[u][v]['w_realized'] -= step_size
    dell_mat = (mat[:, 1:]-mat[:, [0]])/step_size
    return dell_mat, mat[:, 0]

new(G)


100%|██████████| 20000/20000 [00:02<00:00, 8241.63it/s]


(array([[1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [1., 0., 1., 0.],
        ...,
        [1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [0., 1., 0., 1.]]),
 array([0.85381172, 1.47819433, 1.65662586, ..., 4.57410533, 2.28127969,
        1.13166428]))